# Session 1 — Your turn: Attention inside a real model

Write one attention head the way a decoder computes it, check it against PyTorch's own implementation, and see why the scores are divided by the square root of the key size.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

rng = np.random.default_rng(0)
T, d_model, d_head = 6, 8, 4
x = rng.normal(size=(T, d_model))
Wq, Wk, Wv = (rng.normal(size=(d_model, d_head)) for _ in range(3))
from aij_check import check, close, summary

## Task 1: One causal attention head

Write `attention(x, Wq, Wk, Wv)` returning `(weights, output)`: queries, keys and values are `x @ Wq`, `x @ Wk`, `x @ Wv`; the scores are `Q @ K.T / sqrt(d_head)`;
positions after the current one are blocked (set to minus infinity) before a softmax over each row; the output is `weights @ V`.

In [ ]:
def attention(x, Wq, Wk, Wv):
    ...


weights, output = attention(x, Wq, Wk, Wv)

In [ ]:
reference = F.scaled_dot_product_attention(*(torch.tensor(a)[None] for a in (x @ Wq, x @ Wk, x @ Wv)), is_causal=True)[0].numpy()
check("each row of weights sums to 1", lambda: close(weights.sum(axis=1), 1.0, 1e-9))
check("no weight points at a later token", lambda: np.allclose(np.triu(weights, 1), 0))
check("the first token reads only itself", lambda: close(weights[0], [1, 0, 0, 0, 0, 0], 1e-9))
check("the output agrees with PyTorch's scaled_dot_product_attention", lambda: close(output, reference, 1e-9))
summary()

## Task 2: Why divide by the square root of d_k?

For key sizes `d_k` of 4, 64 and 1024, draw 500 random queries and 32 random keys each (standard normal), compute the softmax of the scores with and without the division by `sqrt(d_k)`, and store in `largest_weight` the **average largest weight** per case:
`largest_weight[(d_k, "unscaled")]` and `largest_weight[(d_k, "scaled")]`. Use `np.random.default_rng(1)`.

In [ ]:
gen = np.random.default_rng(1)
largest_weight = {}
for d_k in (4, 64, 1024):
    q, k = gen.normal(size=(500, 1, d_k)), gen.normal(size=(500, 32, d_k))
    raw = (q * k).sum(-1)                         # 500 rows of 32 scores
    ...                                           # fill largest_weight for "unscaled" and "scaled"

In [ ]:
check("six numbers", lambda: len(largest_weight) == 6)
check("without the division the largest weight climbs towards 1 as d_k grows", lambda: largest_weight[(1024, "unscaled")] > 0.9 > largest_weight[(4, "unscaled")] + 0.3)
check("with the division it stays roughly constant", lambda: max(largest_weight[(d, "scaled")] for d in (4, 64, 1024)) - min(largest_weight[(d, "scaled")] for d in (4, 64, 1024)) < 0.08)
check("at d_k = 4 the division matters least", lambda: largest_weight[(4, "unscaled")] - largest_weight[(4, "scaled")] < 0.5 * (largest_weight[(1024, "unscaled")] - largest_weight[(1024, "scaled")]))
summary()